# CS-781 Capstone Project: Phase 2 — Legal Named Entity Recognition (L-NER)
## Champion Hybrid Mixture-of-Experts (MoE) Downstream Adaptation Pipeline

**Key Phase 2 Requirements & Strict Constraints:**
1. **Pretrained Foundation:** Must strictly start from the Phase 1 pretrained Legal Language Model (`HybridMoE`, ~210M parameters). External pretrained models/APIs are strictly banned.
2. **Implementation Constraint:** Raw PyTorch primitives only (`nn.Module`, `nn.MultiheadAttention`, `nn.Linear`). Hugging Face model classes are banned.
3. **Mandatory LoRMA:** Low-Rank Multiplicative Adaptation (Bihany, Patel, Modi, ACL 2025) is **compulsory**.
4. **Mandatory Distillation:** At least 1 knowledge distillation experiment (Teacher -> Student) is **mandatory**.
5. **Primary Metric:** Strict Macro-F1 across 12 legal classes (`APP`, `RESP`, `A.COUNSEL`, `R.COUNSEL`, `JUDGE`, `WIT`, `AUTH`, `COURT`, `STAT`, `PREC`, `DATE`, `CASENO`). Strict offset matching: even a 1-character boundary error is penalized.
6. **Zero-Drift Sliding Window:** Processes long legal documents without exceeding the model's 512-token context limit.


In [ ]:
# Step 1: Environment Setup & Hugging Face Authentication
import os
import sys
import json
import math
import random
import zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Attempt login via Kaggle Secrets or interactive prompt
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Successfully authenticated to Hugging Face via Kaggle Secrets!")
except Exception as e:
    print("Hugging Face login via Kaggle Secrets skipped / not set:", e)


In [ ]:
# Step 1b: Checkpoint Safeguard (Auto-Detects, Reassembles & Clones to /kaggle/working/)
import os
import glob
import shutil
import zipfile

SAVED_CKPT_PATH = "/kaggle/working/champion_moe.pth"

def prepare_checkpoint():
    # 1. Already secured in /kaggle/working/
    if os.path.exists(SAVED_CKPT_PATH) and os.path.getsize(SAVED_CKPT_PATH) > 100 * 1024 * 1024:
        return SAVED_CKPT_PATH

    # 2. Check for unpacked PyTorch checkpoint directory (contains data.pkl)
    data_pkl_matches = glob.glob("/kaggle/input/**/data.pkl", recursive=True)
    if data_pkl_matches:
        ckpt_dir = os.path.dirname(data_pkl_matches[0])
        print(f"[Safeguard] Detected unpacked PyTorch checkpoint directory: {ckpt_dir}")
        print(f"[Safeguard] Packaging into PyTorch format at {SAVED_CKPT_PATH} (takes ~2 seconds)...")
        with zipfile.ZipFile(SAVED_CKPT_PATH, "w", compression=zipfile.ZIP_STORED) as zf:
            for root, dirs, files in os.walk(ckpt_dir):
                for f in files:
                    full_p = os.path.join(root, f)
                    rel_p = os.path.relpath(full_p, ckpt_dir).replace(os.sep, "/")
                    arc_p = rel_p if rel_p.startswith("archive/") else f"archive/{rel_p}"
                    zf.write(full_p, arcname=arc_p)
        size_mb = os.path.getsize(SAVED_CKPT_PATH) / (1024 * 1024)
        print(f"[Safeguard] SUCCESS! Checkpoint assembled at {SAVED_CKPT_PATH} ({size_mb:.1f} MB).")
        return SAVED_CKPT_PATH

    # 3. Check for .pth file anywhere in /kaggle/input/
    for p in glob.glob("/kaggle/input/**/*.pth", recursive=True):
        if os.path.isfile(p) and os.path.getsize(p) > 100 * 1024 * 1024:
            print(f"[Safeguard] Found .pth file at {p}. Copying to {SAVED_CKPT_PATH}...")
            shutil.copy(p, SAVED_CKPT_PATH)
            return SAVED_CKPT_PATH

    return None

ckpt_src = prepare_checkpoint()

if ckpt_src:
    size_mb = os.path.getsize(ckpt_src) / (1024 * 1024)
    print("=" * 65)
    print(f"✅ CHECKPOINT DETECTED & SECURED: {ckpt_src} ({size_mb:.1f} MB)")
    print("=" * 65)
else:
    print("=" * 65)
    print("⚠️ Checkpoint not found yet!")
    print("Current contents of /kaggle/input/:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(f"  {os.path.join(root, f)}")
    print("=" * 65)


In [ ]:
# Step 2: Download Official Phase 2 L-NER Dataset from Hugging Face
from huggingface_hub import hf_hub_download

HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

files_to_download = [
    "Phase_2_LNER/Train/fold1-text.json",
    "Phase_2_LNER/Train/fold1-gold.json",
    "Phase_2_LNER/Train/fold2-text.json",
    "Phase_2_LNER/Train/fold2-gold.json",
    "Phase_2_LNER/Train/labels.txt",
    "Phase_2_LNER/Test/fold3-text.json"
]

data_paths = {}
for rel_path in files_to_download:
    try:
        local_path = hf_hub_download(repo_id=HF_REPO_ID, filename=rel_path, repo_type="dataset")
        data_paths[rel_path] = local_path
        print(f"Downloaded {rel_path} -> {local_path}")
    except Exception as e:
        print(f"Download error for {rel_path}: {e}")


In [ ]:
# Step 3: Inspect Labels and Data Folds
OFFICIAL_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

with open(data_paths["Phase_2_LNER/Train/fold1-text.json"], "r", encoding="utf-8") as f:
    f1_text = json.load(f)
with open(data_paths["Phase_2_LNER/Train/fold1-gold.json"], "r", encoding="utf-8") as f:
    f1_gold = json.load(f)

with open(data_paths["Phase_2_LNER/Train/fold2-text.json"], "r", encoding="utf-8") as f:
    f2_text = json.load(f)
with open(data_paths["Phase_2_LNER/Train/fold2-gold.json"], "r", encoding="utf-8") as f:
    f2_gold = json.load(f)

with open(data_paths["Phase_2_LNER/Test/fold3-text.json"], "r", encoding="utf-8") as f:
    test_text = json.load(f)

# Combine Fold 1 + Fold 2 for complete supervised coverage
combined_text = {**f1_text, **f2_text}
combined_gold = {**f1_gold, **f2_gold}

# Deterministic 10% stratified holdout for validation tracking
all_doc_ids = sorted(list(combined_text.keys()))
val_count = max(8, int(0.10 * len(all_doc_ids)))
val_ids = set(all_doc_ids[:val_count])

train_text = {d: combined_text[d] for d in all_doc_ids if d not in val_ids}
train_gold = {d: combined_gold[d] for d in all_doc_ids if d not in val_ids}

val_text = {d: combined_text[d] for d in val_ids}
val_gold = {d: combined_gold[d] for d in val_ids}

print("=" * 60)
print(f"COMBINED DATASET: {len(combined_text)} annotated documents (Fold 1: {len(f1_text)}, Fold 2: {len(f2_text)})")
print(f"Training Documents: {len(train_text)}")
print(f"Validation Documents (10% holdout): {len(val_text)}")
print(f"Test Documents (Fold 3): {len(test_text)}")
print("=" * 60)

sample_id = list(train_gold.keys())[0]
print(f"\nSample Gold Annotations for Doc {sample_id} (first 3):", train_gold[sample_id][:3])


In [ ]:
# Step 4: Exact Phase 1 Hybrid Mixture-of-Experts Architecture
class StandardExpert(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim)
        )
    def forward(self, x): 
        return self.net(x)

class MoERouter(nn.Module):
    def __init__(self, dim, num_experts):
        super().__init__()
        self.gate = nn.Linear(dim, num_experts, bias=False)
        
    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        weights, indices = torch.max(probs, dim=-1)
        return weights, indices

class MoELayer(nn.Module):
    def __init__(self, dim, num_experts=4):
        super().__init__()
        self.num_experts = num_experts
        self.router = MoERouter(dim, num_experts)
        self.experts = nn.ModuleList([StandardExpert(dim) for _ in range(num_experts)])
        
    def forward(self, x):
        b, seq, d = x.shape
        flat_x = x.view(-1, d)
        weights, indices = self.router(flat_x)
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (indices == i)
            if mask.any():
                exp_out = expert(flat_x[mask])
                out[mask] = exp_out * weights[mask].unsqueeze(-1)
        return out.view(b, seq, d)

class HybridBlock(nn.Module):
    def __init__(self, dim, heads, is_moe=False):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        if is_moe:
            self.ffn = MoELayer(dim, num_experts=4)
        else:
            self.ffn = StandardExpert(dim)
        self.attn_adapter = None
            
    def forward(self, x, mask=None):
        n_x = self.norm1(x)
        attn_out, _ = self.attn(n_x, n_x, n_x, attn_mask=mask)
        if hasattr(self, "attn_adapter") and self.attn_adapter is not None:
            attn_out = self.attn_adapter(attn_out)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x

class HybridMoE(nn.Module):
    def __init__(self, vocab_count=50257, embed_size=768, heads=12, layers=6):
        super().__init__()
        self.embed_size = embed_size
        self.tok_emb = nn.Embedding(vocab_count, embed_size)
        self.pos_emb = nn.Embedding(512, embed_size)
        
        self.blocks = nn.ModuleList()
        for i in range(layers):
            is_moe = (i % 2 != 0)
            self.blocks.append(HybridBlock(embed_size, heads, is_moe))
            
        self.final_norm = nn.LayerNorm(embed_size)
        self.output_head = nn.Linear(embed_size, vocab_count, bias=False)
        self.tok_emb.weight = self.output_head.weight 
        
    def get_hidden_states(self, x, causal=False):
        seq_len = x.shape[1]
        pos = torch.arange(seq_len, device=x.device)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device) * float("-inf"), diagonal=1) if causal else None
        out = self.tok_emb(x) + self.pos_emb(pos)
        for block in self.blocks:
            out = block(out, mask)
        return self.final_norm(out)

    def forward(self, x, causal=True):
        h = self.get_hidden_states(x, causal=causal)
        return self.output_head(h)


In [ ]:
# Step 5: PEFT Adaptation Modules (LoRMA [Compulsory], LoRA, BitFit)
class LoRMAAdapter(nn.Module):
    """Additive Rank Inflation variant (I_+) applied to attention projection."""
    def __init__(self, dim, r=16, alpha=1.0, dropout=0.05):
        super().__init__()
        self.lorma_A = nn.Linear(dim, r, bias=False)
        self.lorma_B = nn.Linear(r, dim, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.alpha = alpha
        nn.init.normal_(self.lorma_A.weight, std=1.0 / math.sqrt(r))
        nn.init.zeros_(self.lorma_B.weight)
        
    def forward(self, x):
        return x + self.lorma_B(self.lorma_A(self.dropout(x))) * self.alpha

class LoRAAdapter(nn.Module):
    """Standard LoRA adapter applied to attention projection."""
    def __init__(self, dim, r=16, alpha=32.0, dropout=0.05):
        super().__init__()
        self.scaling = alpha / r
        self.lora_A = nn.Linear(dim, r, bias=False)
        self.lora_B = nn.Linear(r, dim, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        
    def forward(self, x):
        return x + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

class LoRMALinear(nn.Module):
    """
    LoRMA: Low-Rank Multiplicative Adaptation (Bihany, Patel, Modi, ACL 2025).
    h = (alpha * B * A + I_d) * W_0 * x = W_0(x) + alpha * B(A(W_0(x)))
    """
    def __init__(self, original_linear, r=16, alpha=1.0, dropout=0.05):
        super().__init__()
        self.original_linear = original_linear
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False
            
        self.out_features = original_linear.out_features
        self.in_features = original_linear.in_features
        self.r = r
        self.alpha = alpha
        
        self.lorma_A = nn.Linear(self.out_features, r, bias=False)
        self.lorma_B = nn.Linear(r, self.out_features, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Zero-initialization for exact preservation of pre-trained checkpoint state
        nn.init.normal_(self.lorma_A.weight, std=1.0 / math.sqrt(r))
        nn.init.zeros_(self.lorma_B.weight)

    @property
    def weight(self): return self.original_linear.weight

    @property
    def bias(self): return self.original_linear.bias
        
    def forward(self, x):
        y0 = self.original_linear(x)
        return y0 + self.lorma_B(self.lorma_A(self.dropout(y0))) * self.alpha

class LoRALinear(nn.Module):
    """Standard Additive LoRA (Hu et al., 2021)"""
    def __init__(self, original_linear, r=16, alpha=32.0, dropout=0.05):
        super().__init__()
        self.original_linear = original_linear
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False
            
        self.scaling = alpha / r
        self.lora_A = nn.Linear(original_linear.in_features, r, bias=False)
        self.lora_B = nn.Linear(r, original_linear.out_features, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    @property
    def weight(self): return self.original_linear.weight

    @property
    def bias(self): return self.original_linear.bias
        
    def forward(self, x):
        return self.original_linear(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

class HybridMoEForTokenClassification(nn.Module):
    def __init__(self, backbone, num_labels=25, dropout=0.1, causal=False):
        super().__init__()
        self.backbone = backbone
        self.num_labels = num_labels
        self.causal = causal
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(backbone.embed_size, num_labels)
        
    def forward(self, input_ids):
        h = self.backbone.get_hidden_states(input_ids, causal=self.causal)
        return self.classifier(self.dropout(h))

class CompactLegalStudent(nn.Module):
    """Student Model for Mandatory Knowledge Distillation"""
    def __init__(self, vocab_count=50257, embed_size=384, heads=6, layers=3, num_labels=25):
        super().__init__()
        self.embed_size = embed_size
        self.tok_emb = nn.Embedding(vocab_count, embed_size)
        self.pos_emb = nn.Embedding(512, embed_size)
        self.blocks = nn.ModuleList([HybridBlock(embed_size, heads, is_moe=False) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(embed_size)
        self.classifier = nn.Linear(embed_size, num_labels)
        
    def forward(self, input_ids):
        seq_len = input_ids.shape[1]
        pos = torch.arange(seq_len, device=input_ids.device)
        out = self.tok_emb(input_ids) + self.pos_emb(pos)
        for block in self.blocks:
            out = block(out, mask=None)
        return self.classifier(self.final_norm(out))


In [ ]:
# Step 6: PEFT Injection and Checkpoint Loader Helper
def load_phase1_backbone(checkpoint_path, device="cpu"):
    backbone = HybridMoE(vocab_count=50257, embed_size=768, heads=12, layers=6)
    state_dict = torch.load(checkpoint_path, map_location=device)
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
    elif isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]
    # Strip module. prefix from DataParallel
    clean_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    backbone.load_state_dict(clean_state_dict, strict=False)
    print(f"Successfully loaded Phase 1 backbone weights from: {checkpoint_path}")
    return backbone

def configure_adaptation(model, method="lorma", r=16, alpha=None):
    method = method.lower()
    if alpha is None:
        alpha = 1.0 if method == "lorma" else 32.0
    if method == "full":
        for p in model.parameters(): p.requires_grad = True
    elif method == "linear_head_only":
        for p in model.backbone.parameters(): p.requires_grad = False
        for p in model.classifier.parameters(): p.requires_grad = True
    elif method == "bitfit":
        for n, p in model.named_parameters():
            p.requires_grad = ("bias" in n or "classifier" in n)
    elif method in ["lorma", "lora"]:
        for p in model.backbone.parameters(): p.requires_grad = False
        for p in model.classifier.parameters(): p.requires_grad = True

        dim = model.backbone.embed_size
        for block in model.backbone.blocks:
            if method == "lorma":
                block.attn_adapter = LoRMAAdapter(dim, r=r, alpha=alpha)
                if isinstance(block.ffn, MoELayer):
                    for expert in block.ffn.experts:
                        expert.net[0] = LoRMALinear(expert.net[0], r=r, alpha=alpha)
                        expert.net[2] = LoRMALinear(expert.net[2], r=r, alpha=alpha)
                else:
                    block.ffn.net[0] = LoRMALinear(block.ffn.net[0], r=r, alpha=alpha)
                    block.ffn.net[2] = LoRMALinear(block.ffn.net[2], r=r, alpha=alpha)
            else:
                block.attn_adapter = LoRAAdapter(dim, r=r, alpha=alpha)
                if isinstance(block.ffn, MoELayer):
                    for expert in block.ffn.experts:
                        expert.net[0] = LoRALinear(expert.net[0], r=r, alpha=alpha)
                        expert.net[2] = LoRALinear(expert.net[2], r=r, alpha=alpha)
                else:
                    block.ffn.net[0] = LoRALinear(block.ffn.net[0], r=r, alpha=alpha)
                    block.ffn.net[2] = LoRALinear(block.ffn.net[2], r=r, alpha=alpha)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[PEFT Method: {method.upper()}] Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model


In [ ]:
# Step 7: Zero-Drift Character Offset Alignment & Sliding Window Engine
def build_bio_vocab():
    tag2id = {"O": 0}
    id2tag = {0: "O"}
    idx = 1
    for label in OFFICIAL_LABELS:
        tag2id[f"B-{label}"] = idx; id2tag[idx] = f"B-{label}"; idx += 1
        tag2id[f"I-{label}"] = idx; id2tag[idx] = f"I-{label}"; idx += 1
    return tag2id, id2tag

TAG2ID, ID2TAG = build_bio_vocab()
NUM_CLASSES = len(TAG2ID)

def refine_span(text, start, end, label):
    if start >= end or start >= len(text) or end > len(text): return None
    import string
    s = text[start:end]
    left_diff = len(s) - len(s.lstrip(string.whitespace))
    new_start = start + left_diff
    s_sub = text[new_start:end]
    # Strip only whitespace and punctuation that never terminates entities (, ; :)
    # Preserves legitimate terminations: abbreviations (Ltd., Inc., v.), parentheses (1), etc.
    s_clean = s_sub.rstrip(string.whitespace + ",;:")
    new_end = new_start + len(s_clean)
    if new_start >= new_end: return None
    if (new_end - new_start) < 2: return None
    if (new_end - new_start) > 200: return None  # Hard safeguard against runaway spans
    return [new_start, new_end, label]

def tokens_to_char_spans(token_tags, offsets, raw_text):
    spans = []
    curr_label, s_char, e_char = None, None, None
    for tag_id, (c_start, c_end) in zip(token_tags, offsets):
        if c_start == 0 and c_end == 0: continue
        tag = ID2TAG.get(tag_id, "O")
        if tag.startswith("B-"):
            if curr_label is not None:
                r = refine_span(raw_text, s_char, e_char, curr_label)
                if r: spans.append(r)
            curr_label = tag[2:]; s_char, e_char = c_start, c_end
        elif tag.startswith("I-"):
            lbl = tag[2:]
            if curr_label == lbl:
                e_char = c_end
            else:
                if curr_label is not None:
                    r = refine_span(raw_text, s_char, e_char, curr_label)
                    if r: spans.append(r)
                curr_label, s_char, e_char = lbl, c_start, c_end
        else:
            if curr_label is not None:
                r = refine_span(raw_text, s_char, e_char, curr_label)
                if r: spans.append(r)
            curr_label, s_char, e_char = None, None, None
    if curr_label is not None:
        r = refine_span(raw_text, s_char, e_char, curr_label)
        if r: spans.append(r)
    return spans

def merge_and_deduplicate_spans(spans, raw_text):
    if not spans: return []
    if isinstance(raw_text, dict):
        raw_text = raw_text.get("text", raw_text.get("raw_text", str(raw_text)))
    else:
        raw_text = str(raw_text)
    refined = []
    for s in spans:
        r = refine_span(raw_text, s[0], s[1], s[2])
        if r: refined.append(r)
    if not refined: return []
    spans = sorted(refined, key=lambda x: (x[0], -(x[1] - x[0])))
    
    # Strict deduplication & non-overlapping boundary resolution
    final_spans = []
    for s in spans:
        if any(m[0] == s[0] and m[1] == s[1] and m[2] == s[2] for m in final_spans): continue
        if final_spans and s[0] < final_spans[-1][1]: continue
        final_spans.append(s)
    return final_spans

def decode_tokens_calibrated(logits, entity_threshold=0.50):
    probs = F.softmax(logits, dim=-1)
    p_entity = 1.0 - probs[..., 0]
    best_entity_tags = torch.argmax(logits[..., 1:], dim=-1) + 1
    preds = torch.where(p_entity >= entity_threshold, best_entity_tags, torch.zeros_like(best_entity_tags))
    return preds.cpu().numpy()

def normalize_gold_spans(raw_spans):
    if not raw_spans: return []
    if isinstance(raw_spans, dict):
        for k in ["spans", "entities", "annotations"]:
            if k in raw_spans and isinstance(raw_spans[k], list):
                raw_spans = raw_spans[k]; break
    normalized = []
    for item in raw_spans:
        if isinstance(item, (list, tuple)):
            if len(item) >= 3:
                start, end, label = item[0], item[1], item[2]
            else: continue
        elif isinstance(item, dict):
            start = item.get("start", item.get("start_char", None))
            end = item.get("end", item.get("end_char", None))
            label = item.get("label", item.get("entity", item.get("tag", None)))
        else: continue
        if start is not None and end is not None and label is not None:
            try:
                s, e, l = int(start), int(end), str(label).strip()
                if s < e and l in OFFICIAL_LABELS:
                    normalized.append([s, e, l])
            except (ValueError, TypeError):
                continue
    return sorted(normalized, key=lambda x: x[0])

def align_gold_to_tokens(offsets, gold_spans):
    labels = [0] * len(offsets)
    clean_spans = normalize_gold_spans(gold_spans)
    for g_start, g_end, g_lbl in clean_spans:
        b_tag = TAG2ID[f"B-{g_lbl}"]
        i_tag = TAG2ID[f"I-{g_lbl}"]
        first = False
        for i, (ts, te) in enumerate(offsets):
            if ts == 0 and te == 0: continue
            if max(ts, g_start) < min(te, g_end):
                labels[i] = b_tag if not first else i_tag
                first = True
    return labels

def chunk_document(doc_id, text, tokenizer, gold_spans=None, max_length=512, stride=256):
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    input_ids, offsets = enc["input_ids"], enc["offset_mapping"]
    labels = align_gold_to_tokens(offsets, gold_spans) if gold_spans is not None else [-100] * len(input_ids)
    chunks = []
    total_tokens = len(input_ids)
    for s in range(0, max(1, total_tokens), stride):
        e = min(s + max_length, total_tokens)
        c_ids, c_offs, c_lbls = input_ids[s:e], offsets[s:e], labels[s:e]
        valid_len = len(c_ids)
        pad = max_length - valid_len
        c_ids = c_ids + [tokenizer.eos_token_id] * pad
        c_lbls = c_lbls + [-100] * pad
        c_offs = c_offs + [(0, 0)] * pad
        chunks.append({
            "doc_id": doc_id, "input_ids": c_ids, "labels": c_lbls,
            "offset_mapping": c_offs, "raw_text": text,
            "start_idx": s, "end_idx": e, "valid_len": valid_len
        })
        if e == total_tokens: break
    return chunks

class ChunkedDataset(Dataset):
    def __init__(self, text_dict, gold_dict, tokenizer, max_length=512, stride=256):
        self.chunks = []
        for d_id, txt in text_dict.items():
            if isinstance(txt, dict):
                txt = txt.get("text", txt.get("raw_text", str(txt)))
            golds = gold_dict.get(d_id, []) if gold_dict is not None else None
            self.chunks.extend(chunk_document(d_id, txt, tokenizer, golds, max_length, stride))
    def __len__(self): return len(self.chunks)
    def __getitem__(self, idx):
        c = self.chunks[idx]
        return {
            "doc_id": c["doc_id"],
            "input_ids": torch.tensor(c["input_ids"], dtype=torch.long),
            "labels": torch.tensor(c["labels"], dtype=torch.long),
            "offsets": c["offset_mapping"],
            "text": c["raw_text"],
            "start_idx": c["start_idx"],
            "end_idx": c["end_idx"],
            "valid_len": c["valid_len"]
        }

def collate_fn(batch):
    return {
        "doc_ids": [b["doc_id"] for b in batch],
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "offsets": [b["offsets"] for b in batch],
        "texts": [b["text"] for b in batch],
        "start_indices": [b["start_idx"] for b in batch],
        "end_indices": [b["end_idx"] for b in batch],
        "valid_lengths": [b["valid_len"] for b in batch]
    }

def predict_documents_pooled(model, data_loader, text_dict, tokenizer, dev, entity_threshold=0.50):
    model.eval()
    doc_accum = {}
    for doc_id, raw_t in text_dict.items():
        if isinstance(raw_t, dict):
            raw_t = raw_t.get("text", raw_t.get("raw_text", str(raw_t)))
        enc = tokenizer(raw_t, return_offsets_mapping=True, add_special_tokens=False)
        doc_len = len(enc["input_ids"])
        doc_accum[doc_id] = {
            "logits": torch.zeros((doc_len, NUM_CLASSES), dtype=torch.float32),
            "counts": torch.zeros((doc_len, 1), dtype=torch.float32),
            "offsets": enc["offset_mapping"],
            "text": raw_t
        }

    with torch.no_grad():
        for b in data_loader:
            x = b["input_ids"].to(dev)
            batch_logits = model(x).cpu()
            for i, did in enumerate(b["doc_ids"]):
                if did not in doc_accum: continue
                s = b["start_indices"][i]
                e = b["end_indices"][i]
                v_len = b["valid_lengths"][i]
                chunk_lg = batch_logits[i, :v_len]
                doc_accum[did]["logits"][s:e] += chunk_lg[:e-s]
                doc_accum[did]["counts"][s:e] += 1.0

    final_preds = {}
    for did, data in doc_accum.items():
        counts = torch.clamp(data["counts"], min=1.0)
        avg_logits = data["logits"] / counts
        tags = decode_tokens_calibrated(avg_logits.unsqueeze(0), entity_threshold=entity_threshold)[0]
        spans = tokens_to_char_spans(tags.tolist(), data["offsets"], data["text"])
        final_preds[did] = merge_and_deduplicate_spans(spans, data["text"])
    return final_preds


In [ ]:
# Step 8: Strict Macro-F1 Evaluator
def evaluate_strict(gold_dict, pred_dict):
    tp_c = {c: 0 for c in OFFICIAL_LABELS}
    fp_c = {c: 0 for c in OFFICIAL_LABELS}
    fn_c = {c: 0 for c in OFFICIAL_LABELS}
    
    for d_id in set(gold_dict.keys()).union(set(pred_dict.keys())):
        g_raw = gold_dict.get(d_id, [])
        g_spans = [(g[0], g[1], g[2]) for g in normalize_gold_spans(g_raw)]
        p_raw = pred_dict.get(d_id, [])
        p_spans = [(p[0], p[1], p[2]) for p in normalize_gold_spans(p_raw)]
        m_g, m_p = set(), set()
        for pi, p in enumerate(p_spans):
            for gi, g in enumerate(g_spans):
                if gi in m_g: continue
                if p[0] == g[0] and p[1] == g[1] and p[2] == g[2]:
                    m_g.add(gi); m_p.add(pi); tp_c[p[2]] += 1; break
        for pi, p in enumerate(p_spans):
            if pi not in m_p: fp_c[p[2]] += 1
        for gi, g in enumerate(g_spans):
            if gi not in m_g: fn_c[g[2]] += 1

    f1s = []
    for c in OFFICIAL_LABELS:
        tp, fp, fn = tp_c[c], fp_c[c], fn_c[c]
        p = tp / (tp + fp) if tp + fp > 0 else 0.0
        r = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if p + r > 0 else 0.0
        f1s.append(f1)
    macro_f1 = sum(f1s) / len(f1s)
    return macro_f1, f1s


In [ ]:
# Step 9: Configure & Run Experiment
# SELECT YOUR EXPERIMENT METHOD: 'lorma' (compulsory), 'lora', 'full', 'bitfit', 'distill'
EXPERIMENT_METHOD = "lorma"
EPOCHS = 12
BATCH_SIZE = 8
LEARNING_RATE = 7e-4 if EXPERIMENT_METHOD in ["lorma", "lora"] else 5e-5

import gc
import glob

# Clean GPU memory between runs
gc.collect()
torch.cuda.empty_cache()

# -------------------------------------------------------------
# 1. Automatic Checkpoint Discovery & Verification
# -------------------------------------------------------------
SAVED_CKPT_PATH = "/kaggle/working/champion_moe.pth"

import zipfile
import shutil

def locate_phase1_checkpoint():
    if os.path.exists(SAVED_CKPT_PATH) and os.path.getsize(SAVED_CKPT_PATH) > 100 * 1024 * 1024:
        return SAVED_CKPT_PATH
        
    data_pkl_matches = glob.glob("/kaggle/input/**/data.pkl", recursive=True)
    if data_pkl_matches:
        ckpt_dir = os.path.dirname(data_pkl_matches[0])
        print(f"[Safeguard] Packaging unpacked checkpoint directory: {ckpt_dir} ...")
        with zipfile.ZipFile(SAVED_CKPT_PATH, "w", compression=zipfile.ZIP_STORED) as zf:
            for root, dirs, files in os.walk(ckpt_dir):
                for f in files:
                    full_p = os.path.join(root, f)
                    rel_p = os.path.relpath(full_p, ckpt_dir).replace(os.sep, "/")
                    arc_p = rel_p if rel_p.startswith("archive/") else f"archive/{rel_p}"
                    zf.write(full_p, arcname=arc_p)
        return SAVED_CKPT_PATH
        
    for p in glob.glob("/kaggle/input/**/*.pth", recursive=True):
        if os.path.isfile(p) and os.path.getsize(p) > 100 * 1024 * 1024:
            return p
            
    return None

PHASE1_CKPT = locate_phase1_checkpoint()

if PHASE1_CKPT is None:
    raise FileNotFoundError(
        "Could not find Phase 1 checkpoint in /kaggle/working/ or /kaggle/input/!"
    )

print("=" * 60)
print("PHASE 1 PRETRAINED BACKBONE READY:")
print(f"Path: {PHASE1_CKPT}")
print(f"Size: {os.path.getsize(PHASE1_CKPT) / (1024 * 1024):.2f} MB")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

train_ds = ChunkedDataset(train_text, train_gold, tokenizer)
val_ds = ChunkedDataset(val_text, val_gold, tokenizer)
test_ds = ChunkedDataset(test_text, None, tokenizer)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Compute Balanced Class Weights (Preserve O background token stability)
print("[Weights] Computing balanced inverse-frequency class weights (power=0.15, clamp=5.0)...")
tag_counts = torch.zeros(NUM_CLASSES, dtype=torch.float)
for c in train_ds.chunks:
    for lbl in c["labels"]:
        if 0 <= lbl < NUM_CLASSES:
            tag_counts[lbl] += 1.0
tag_counts = torch.clamp(tag_counts, min=1.0)
class_weights = (tag_counts.max() / tag_counts) ** 0.15
class_weights[0] = 1.0  # Preserve full weight for background tokens!
class_weights = torch.clamp(class_weights, max=5.0)
class_weights = (class_weights / class_weights.mean()).to(device)

# Initialize and Load Pretrained Backbone
backbone = load_phase1_backbone(PHASE1_CKPT, device="cpu")

model = HybridMoEForTokenClassification(backbone, num_labels=NUM_CLASSES)
model = configure_adaptation(model, method=EXPERIMENT_METHOD, r=16, alpha=1.0)
model.to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

best_f1 = 0.0
best_threshold = 0.50

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, b in enumerate(train_loader):
        x = b["input_ids"].to(device)
        y = b["labels"].to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, NUM_CLASSES), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if step % 25 == 0:
            print(f"Epoch [{epoch}/{EPOCHS}] Step [{step}/{len(train_loader)}] Loss: {loss.item():.4f} LR: {scheduler.get_last_lr()[0]:.2e}")
            
    # Validation: evaluate across candidate confidence thresholds via Document-Level Logit Pooling
    for thresh in [0.45, 0.50, 0.55, 0.60]:
        val_preds = predict_documents_pooled(model, val_loader, val_text, tokenizer, device, entity_threshold=thresh)
        val_f1, _ = evaluate_strict(val_gold, val_preds)
        print(f"Epoch {epoch} | Thresh {thresh:.2f} | Val Strict Macro-F1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_threshold = thresh
            save_obj = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(save_obj, f"best_model_{EXPERIMENT_METHOD}.pth")
            print(f"🔥 New Best Model Saved! F1: {best_f1:.4f} (Thresh: {thresh:.2f})")

print("")
print(f"============================================================")
print(f"Training Complete! Best Val F1: {best_f1:.4f} at Threshold: {best_threshold:.2f}")
print(f"============================================================")


In [ ]:
# Step 10: Generate Final Test Predictions & Create submission.zip
print(f"Running Document-Level Pooled Test Inference on {len(test_text)} documents...")

# Load best checkpoint if saved
best_ckpt = f"best_model_{EXPERIMENT_METHOD}.pth"
if os.path.exists(best_ckpt):
    print(f"Loading best checkpoint from {best_ckpt} (Val F1: {best_f1:.4f}, Thresh: {best_threshold:.2f})...")
    load_target = model.module if hasattr(model, "module") else model
    load_target.load_state_dict(torch.load(best_ckpt, map_location=device), strict=False)

final_test_predictions = predict_documents_pooled(model, test_loader, test_text, tokenizer, device, entity_threshold=best_threshold)

total_test_spans = sum(len(sp) for sp in final_test_predictions.values())
avg_spans = total_test_spans / max(1, len(final_test_predictions))
span_lens = [s[1] - s[0] for sp in final_test_predictions.values() for s in sp] or [0]
avg_len = sum(span_lens) / max(1, len(span_lens))

print("=" * 60)
print(f"DOCUMENT-POOLED TEST PREDICTION STATISTICS:")
print(f"Total Predicted Spans: {total_test_spans}")
print(f"Average Spans / Document: {avg_spans:.1f}")
print(f"Average Span Length: {avg_len:.1f} characters")
print("=" * 60)

# Save prediction.json
with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(final_test_predictions, f, indent=2)

# Pack into submission.zip
with zipfile.ZipFile("submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("prediction.json", arcname="prediction.json")

print("")
print("SUCCESS! Created submission.zip with prediction.json at archive root.")
print(f"Total documents predicted: {len(final_test_predictions)}")
sample_test_id = list(final_test_predictions.keys())[0]
print(f"Sample predictions for {sample_test_id}:", final_test_predictions[sample_test_id][:5])
